Copyright 2026 Google LLC.

In [2]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Priority and Flex Inference Tiers

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Inference_tiers.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The Gemini API offers different `service_tiers` to help you manage cost and latency.

The **Priority** and **Flex** tiers allow you to route background jobs to Flex and interactive jobs to Priority using standard synchronous endpoints. This eliminates the complexity of async job management while giving you the economic and performance benefits of specialized tiers:

- **[Priority](https://ai.google.dev/gemini-api/docs/priority-inference) (`"priority"`):** Millisecond latency for critical apps (+75-100% cost). Traffic is strictly non-sheddable.
- **[Flex](https://ai.google.dev/gemini-api/docs/flex-inference) (`"flex"`):** 1-15 min target latency for background tasks (-50% cost). Fully synchronous, but uses opportunistic compute.

**In this notebook, you will learn:**
1.  Comparison of Priority and Flex
2.  How to use Priority tier
3.  How to use Flex tier
4.  How to adjust timeouts
5.  How to implement retries

> **Note:** Inference tiers are paid only features. Flex is available for all tiers on the paid tier, and Priority is available for Tier 2 & 3. This notebook won't run with the Free Tier. (cf. [pricing](https://ai.google.dev/pricing) for more details).

> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/Inference_tiers.ipynb).

# Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai).

In [ ]:
%pip install -U -q "google-genai>=2.9.0"  # 2.0 for Interactions API

Note: you may need to restart the kernel to use updated packages.


### Setup your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for a walkthrough.

In [8]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

Initialize a client with your API key:

In [10]:
from google import genai
from google.genai import errors, types

client = genai.Client(api_key=GEMINI_API_KEY)

### Choose a model

Most Gemini 2.5 and Gemini 3 models support inference tiers. Refer to the documentation for more details:
- [Flex supported models](https://ai.google.dev/gemini-api/docs/flex-inference#supported-models)
- [Priority supported models](https://ai.google.dev/gemini-api/docs/priority-inference#supported-models)

In [12]:
MODEL_ID = "gemini-3.6-flash" # @param ["gemini-3.5-flash-lite", "gemini-3.6-flash", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

## Priority and Flex comparison

| Feature | Standard | Flex | Priority | Batch | Caching |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Pricing** | Full Price | 50% discount | 75% to 100% more than standard | 50% discount | 90% discount + Prorated token storage |
| **Latency** | Seconds to minutes | Minutes (1–15 min target) | Seconds | Up to 24 hours | Faster time-to-first-token |
| **Reliability** | High / Medium-high | Best-effort (Sheddable) | High (Non-sheddable) | High (for throughput) | N/A |
| **Interface** | Synchronous | Synchronous | Synchronous | Asynchronous | Saved state |
| **Best use case** | General application workflows | Non-urgent sequential chains | Production, user-facing apps | Massive datasets, offline evals | Recurring queries over same file |

### Key benefits of Priority

* **Low latency**: Designed for second response times for interactive,
user-facing AI tools.
* **High reliability**: Traffic is treated with the highest criticality and is
strictly non-sheddable.
* **Graceful degradation**: Traffic spikes exceeding dynamic limits are
automatically downgraded to the Standard tier for processing instead of failing,
preventing service outages.

### Key benefits of Flex

* **Cost efficiency**: Substantial savings for non-production evals, background agents, and data enrichment.
* **Low friction**: No need to manage batch objects, job IDs, or polling; simply add a single parameter to your existing requests.
* **Synchronous workflows**: Ideal for sequential API chains where the next request depends on the output of the previous one, making it more flexible than Batch for agentic workflows.

## Priority inference

The Gemini Priority API is a premium inference tier designed for
business-critical workloads that require lower latency and the highest
reliability at a premium price point. Priority tier traffic is prioritized above
standard API and Flex tier traffic.

### How Priority works

Priority inference routes requests to high-criticality compute queues, offering
predictable, fast performance for user-facing applications. Its primary
mechanism is a graceful server-side downgrade to standard processing for traffic
that exceeds dynamic limits, ensuring application stability instead of failing
the request.

### Priority rate limits

Priority consumption holds its own rate
limits even though consumption is counted towards overall interactive traffic
rate limits. **Default rate limits are: 0.3x the [standard rate limit](https://aistudio.google.com/rate-limit) for each model and tier**.

### How to use Priority

To use the Priority tier, set the `service_tier` field in the request body to
`"priority"`. The default tier is standard if the field is omitted.

In [15]:
try:
    interaction = client.interactions.create(
        model=MODEL_ID,
        input="Why is the sky blue?",
        service_tier='priority',
    )

    # Check the service tier from the interaction metadata
    print(f"Service tier: {interaction.service_tier}")
    print(interaction.steps[-1].content[0].text)

except errors.APIError as e:
    print(e.code, e.message)
except Exception as e:
    print(f"Error during API call: {e}")

Service tier: priority
The sky is blue because of a phenomenon called **Rayleigh scattering**. 

Here is the step-by-step breakdown of how it works:

### 1. Sunlight is a rainbow
Although sunlight looks white to us, it is actually made up of all the colors of the rainbow (red, orange, yellow, green, blue, indigo, and violet). Light travels as **waves**, and each color has a different wavelength. Red light travels in long, lazy waves, while blue and violet light travel in much shorter, choppier waves.

### 2. The Atmosphere is an obstacle course
Earth’s atmosphere is filled with gases (mostly nitrogen and oxygen) and particles. When sunlight hits the atmosphere, it strikes the molecules of these gases.

### 3. Short waves scatter more
Because **blue light** travels in shorter, smaller waves, it strikes the gas molecules and gets scattered in every direction. Red light, with its long wavelengths, passes through the atmosphere more easily without bumping into as many molecules. 

Because 

You'll find the service tier in the headers:

In [17]:
# The service tier is available directly on the interaction object
print(f"Service tier: {interaction.service_tier}")

Service tier: priority


## Flex inference

The Gemini Flex API is an inference tier that offers a 50% cost reduction
compared to standard rates, in exchange for variable latency and best-effort
availability. It's designed for latency-tolerant workloads that require
synchronous processing but don't need the real-time performance of the standard
API.

### How Flex works

Gemini Flex inference bridges the gap between the standard API and the 24-hour
turnaround of the [Batch API](https://ai.google.dev/gemini-api/docs/batch-api). It utilizes off-peak,
"sheddable" compute capacity to provide a cost-effective solution for background
tasks and sequential workflows.

### How to use Flex

To use the Flex tier, specify the `service_tier` as `"flex"` in the
request body. By default, requests use the standard tier if this field is
omitted.


In [19]:
try:
    interaction = client.interactions.create(
        model=MODEL_ID,
        input="Why is the sky blue?",
        service_tier='flex',
    )

    print(interaction.steps[-1].content[0].text)
except errors.APIError as e:
    print(e.code, e.message)

The sky is blue because of a phenomenon called **Rayleigh scattering**. Here is the step-by-step breakdown of how it works:

### 1. Sunlight is a rainbow
Although sunlight looks white to us, it is actually made up of all the colors of the rainbow (red, orange, yellow, green, blue, indigo, and violet). Light travels as waves, and each color has a different wavelength. Red light has long, lazy waves, while blue and violet light have much shorter, choppier waves.

### 2. The Atmosphere is an obstacle course
Earth’s atmosphere is filled with gases (mostly nitrogen and oxygen). When sunlight hits the atmosphere, it strikes the molecules of these gases.

### 3. Short waves scatter easily
Because blue light travels in shorter, smaller waves, it strikes the gas molecules and gets scattered in every direction. This is **Rayleigh scattering**. 

Imagine throwing a bunch of large balls (red light) and small marbles (blue light) through a forest of thin trees. The large balls might sail straight t

In [20]:
print(f"Service tier: {interaction.service_tier}")

Service tier: flex


### Adjusting timeout windows

You can configure per-request timeouts for the REST API and client libraries,
and global timeouts only when using the client libraries.

Always ensure your client-side timeout covers the intended server patience
window (e.g., 600s+ for Flex wait queues). The SDKs expect timeout values in
milliseconds.

#### Per-request timeouts


In [22]:
try:
    interaction = client.interactions.create(
        model=MODEL_ID,
        input="why is the sky blue?",
        service_tier="flex",
        timeout=900,
    )
    print(interaction.steps[-1].content[0].text)
except errors.APIError as e:
    print(e.code, e.message)

The sky is blue because of a phenomenon called **Rayleigh scattering**.

Here is the step-by-step breakdown of how it works:

### 1. Sunlight is a mix of all colors
Although sunlight looks white to us, it is actually made up of all the colors of the rainbow (red, orange, yellow, green, blue, indigo, and violet). Light travels as **waves**, and each color has a different wavelength. Red light has long, lazy waves, while blue and violet light have short, choppy waves.

### 2. The Atmosphere is an obstacle course
Earth’s atmosphere is filled with gases (mostly nitrogen and oxygen). When sunlight hits the atmosphere, it strikes the molecules of these gases.

### 3. Short waves scatter more
Because gas molecules are very small, they affect the colors differently:
*   The **longer wavelengths** (reds and yellows) pass through the atmosphere mostly straight, without bumping into much.
*   The **shorter wavelengths** (blues and violets) strike the gas molecules and get scattered in every direc

Example with streaming (timeout applies to each chunk or overall, depending on implementation)

In [24]:
try:
    response = client.interactions.create(
        model=MODEL_ID,
        input="List 5 ideas for a sci-fi movie.",
        service_tier="flex",
        stream=True,
    )
    for event in response:
        if hasattr(event, 'delta') and hasattr(event.delta, 'text') and event.delta.text:
            print(event.delta.text, end="")
except errors.APIError as e:
    print(e.code, e.message)
except Exception as e:
    print(f"An error occurred during streaming: {e}")

-future thrillers:

### 1. The Echo Chamber
**Genre:** Psychological Sci-Fi / Mystery
**The Premise:** In the near future, a technology called "Echo" allows people to revisit their own memories as a fully immersive VR experience. However, users are strictly forbidden from interacting with their past selves. 
**The Plot:** An Echo technician who spends his days "cleaning" the digital memories of the wealthy discovers a recurring glitch: a woman who shouldn't exist is appearing in the background of dozens of different people’s memories across the span of 30 years. When he realizes she is looking directly at the "camera" (the user's eyes), he must break the law to enter his own past and find out how she is hijacking the timeline of human consciousness.

### 2. Gravity’s Debt
**Genre:** Space Western / Political Thriller
**The Premise:** Earth has become a corporate-owned wasteland. To leave, citizens must sign "Gravity Contracts," working on asteroid mines until they pay off the cost of t

#### Global timeouts

If you want all API calls made through a specific `genai.Client` instance
(client libraries only) to have a default timeout, you can configure this when
initializing the client using `http_options` and `genai.types.HttpOptions`.

In [26]:
global_timeout_ms = 120000

client_with_global_timeout = genai.Client(
    api_key=GEMINI_API_KEY,
    http_options=types.HttpOptions(timeout=global_timeout_ms)
)

try:
    # Calling interactions.create using global timeout...
    interaction = client_with_global_timeout.interactions.create(
        model=MODEL_ID,
        input="Summarize the history of AI development since 2000.",
        service_tier="flex",
    )

    print(interaction.steps[-1].content[0].text)
except Exception as e:
    print(f"Error: {e}")

The history of AI since 2000 is a story of moving from niche academic research and "expert systems" to a ubiquitous technology that powers global infrastructure. This development can be broken down into four distinct eras.

---

### 1. The Pre-Deep Learning Era (2000–2011)
At the turn of the millennium, AI was emerging from the "AI Winter" of the 1990s. The focus was on **Statistical Machine Learning.**

*   **Logic to Probability:** Instead of trying to program every rule of human logic, researchers began using statistics to help computers "guess" outcomes based on data.
*   **Big Data Begins:** The rise of the internet provided the massive datasets (via Google, Amazon, and Yahoo) necessary for training algorithms.
*   **Key Milestones:** 
    *   **2002:** The Roomba brings autonomous AI into the home.
    *   **2004/2005:** The DARPA Grand Challenges kickstart the race for self-driving cars.
    *   **2011:** **IBM Watson** wins *Jeopardy!*, proving that AI could handle complex natu

### Implementing retries

Because Flex is sheddable and fails with 503 errors, here is an example of
optionally implementing retry logic to continue with failed requests:

In [28]:
import time


def call_with_retry(max_retries=3, base_delay=5):
    for attempt in range(max_retries):
        try:
            return client.interactions.create(
                model=MODEL_ID,
                input="Provide a very brief definition of machine learning.",
                service_tier="flex",
            )
        except errors.APIError as e:
            # Check for 503 Service Unavailable or 429 Rate Limits
            print(e.code)
            if attempt < max_retries - 1:
                delay = base_delay * (2**attempt)  # Exponential Backoff
                print(f"Flex busy, retrying in {delay}s...")
                time.sleep(delay)
            else:
                # Fallback to standard on last strike
                print("Flex exhausted, falling back to Standard...")
                return client.interactions.create(
                    model=MODEL_ID,
                    input="Provide a very brief definition of machine learning.",
                )
        except Exception as e:
            print(f"An error occurred: {e}")

# Usage
interaction = call_with_retry()
print(interaction.steps[-1].content[0].text)

Machine learning is a subset of AI where computers use data and algorithms to learn and improve at tasks without being explicitly programmed.


## Further resources

To learn more, see the following resources:

- [Optimization docs](https://ai.google.dev/gemini-api/docs/optimization)
- [Priority docs](https://ai.google.dev/gemini-api/docs/priority-inference)
- [Flex docs](https://ai.google.dev/gemini-api/docs/flex-inference)